In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler, RobustScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math
from sklearn.ensemble import IsolationForest
import shap
from sklearn.manifold import TSNE

import warnings
warnings.filterwarnings('ignore')

import sys
import os

sys.path.insert(0, os.path.abspath(".."))
from scaling import scale_features

from config import load_data, FEATURES, SKEWED, FRAUD_IDS

In [85]:
def test_model(
    df,
    features,
    skewed,
    model = 'iso',
    top = 20,
    ):
    print('Starting test_model')
    df_copy = df.copy()
    X_std = scale_features(data=df_copy, scaler_type="standard", features=features, skewed=skewed)
    train_data = X_std[~X_std.index.isin(FRAUD_IDS)]
    
    if model == 'ocsvm':
        ocsvm = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale")
        ocsvm.fit(train_data)
        df_copy['anomaly_score_std'] = ocsvm.decision_function(X_std)
        df_copy['anomaly_label_std'] = ocsvm.predict(X_std)
        hit_rate = df_copy[df_copy['is_fraud']]["anomaly_label_std"].value_counts(normalize=True).get(-1, 0)
        top_n = df_copy['anomaly_score_std'].sort_values().head(top)
        fraud_in_top_n = top_n.index.isin(FRAUD_IDS)
        n_fraud = int(df_copy['is_fraud'].sum())
        recall_at_n = fraud_in_top_n.sum() / n_fraud if n_fraud else np.nan
        precision_at_n = fraud_in_top_n.mean()
        table = pd.DataFrame({
            'score': top_n.values,
            'is_fraud': fraud_in_top_n
        }, index=top_n.index)
        print(f"One Class SVM: Anomaly Hit Rate = {hit_rate:.4f}, recall@{top} = {recall_at_n:.4f}, precision@{top} = {precision_at_n:.4f}")
        return table, df_copy, top_n, X_std
    if model == 'iso':
        iso = IsolationForest(
            n_estimators=100,
            contamination=0.001,
            random_state=42
        )
        iso.fit(train_data)
        df_copy['anomaly_score_std'] = iso.decision_function(X_std)
        df_copy['anomaly_label_std'] = iso.predict(X_std)
        hit_rate = df_copy[df_copy['is_fraud']]["anomaly_label_std"].value_counts(normalize=True).get(-1, 0)
        top_n = df_copy['anomaly_score_std'].sort_values().head(top)
        fraud_in_top_n = top_n.index.isin(FRAUD_IDS)
        n_fraud = int(df_copy['is_fraud'].sum())
        recall_at_n = fraud_in_top_n.sum() / n_fraud if n_fraud else np.nan
        precision_at_n = fraud_in_top_n.mean()
        table = pd.DataFrame({
            'score': top_n.values,
            'is_fraud': fraud_in_top_n
        }, index=top_n.index)
        print(f"Isolation Forest: Anomaly Hit Rate = {hit_rate:.4f}, recall@{top} = {recall_at_n:.4f}, precision@{top} = {precision_at_n:.4f}")
        return table, df_copy, top_n, X_std

def check_top_anomaly(
    df,
    X_std,
    top_n,
    features
    ):
    fraud_ids = df[df.index.isin(top_n.index)].index
    # Sort fraud_ids by anomaly score (ascending: most anomalous first)
    fraud_ids_sorted = df.loc[fraud_ids, 'anomaly_score_std'].sort_values().index

    # Collect percentiles for each fraud person_id (for selected features)
    train = X_std[~X_std.index.isin(FRAUD_IDS)]

    features_for_table = ['anomaly_score_std'] + features
    rows = []

    for person_id in fraud_ids_sorted:
        row = {}
        for feat in features_for_table:
            if feat == 'anomaly_score_std':
                row[feat] = round(df.loc[person_id, 'anomaly_score_std'], 4)
            else:
                val = X_std.loc[person_id, feat]
                # Calculate percentile vs train
                percentile = (train[feat] < val).mean() * 100
                row[feat] = round(percentile, 2)
        rows.append(row)

    result_df = pd.DataFrame(rows, index=fraud_ids_sorted)
    result_df.index.name = "person_id"
    result_df.to_csv('res_df.csv')
    return result_df

def check_real_frauds(
    df,
    X_std,
    top_n,
    features
):
    fraud_ids = df[df['is_fraud'] == 1].index
    fraud_ids_sorted = df.loc[fraud_ids, 'anomaly_score_std'].sort_values().index

    train = X_std[~X_std.index.isin(FRAUD_IDS)]

    features_for_table = ['anomaly_score_std'] + features
    rows = []

    for person_id in fraud_ids_sorted:
        row = {}
        for feat in features_for_table:
            if feat == 'anomaly_score_std':
                row[feat] = round(df.loc[person_id, 'anomaly_score_std'], 4)
            else:
                val = X_std.loc[person_id, feat]
                percentile = (train[feat] < val).mean() * 100
                row[feat] = round(percentile, 2)
        rows.append(row)

    result_df = pd.DataFrame(rows, index=fraud_ids_sorted)
    result_df.index.name = "person_id"
    result_df.to_csv('res_df.csv')
    return result_df

### 1. використовую усі фічі, без жодних обмежень по активності користувача  
Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.1429, Recall = 0.0000

In [35]:
df, client_data = load_data(activity_state = 1)
X_std = scale_features(data=client_data, scaler_type="standard")

features = [
    'num_of_trn',
    'days_visits',
    'gross_amount_mean',
    'gross_amount_sum',
    'bonuses_accum_sum',
    'bonuses_used_sum',
    'num_of_waiters',
    'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'share_top_waiter',
    'share_bonus_trn',
    'share_bonus_after_first',
    'num_of_places',
    'share_top_places',
    'share_bonuses_used_top_waiter'
]
skewed = [
    'num_of_trn',
    'days_visits',
    'gross_amount_mean',
    'gross_amount_sum',
    'bonuses_accum_sum',
    'bonuses_used_sum',
    'num_of_waiters',
    'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'num_of_places'
]

X_std2 =  scale_features(
        data = client_data,
        scaler_type = "standard",
        features = features,
        skewed = skewed,
        show_charts = False,
        fit_data = None,
)
iso = IsolationForest(
    n_estimators=100,
    contamination=0.001,
    random_state=42
)

train_data = X_std2[~X_std2.index.isin(FRAUD_IDS)]
iso.fit(train_data)

client_data['anomaly_score_std'] = iso.decision_function(X_std2)
client_data['anomaly_label_std'] = iso.predict(X_std2)
hit_rate = client_data[client_data['is_fraud']]["anomaly_label_std"].value_counts(normalize=True).get(-1, 0)

top_20 = client_data['anomaly_score_std'].sort_values().head(20)
fraud_in_top_20 = top_20.index.isin(FRAUD_IDS)
n_fraud = int(client_data['is_fraud'].sum())
# recall@20 = (frauds in top 20) / total_frauds (same definition as models.py)
recall_at_20 = fraud_in_top_20.sum() / n_fraud if n_fraud else np.nan
# precision@20 = (frauds in top 20) / 20 (what the notebook previously called "Recall")
precision_at_20 = fraud_in_top_20.mean()
table = pd.DataFrame({
    'score': top_20.values,
    'is_fraud': fraud_in_top_20
}, index=top_20.index)
print(f"Isolation Forest with contamination=0.001: Anomaly Hit Rate = {hit_rate:.4f}, recall@20 = {recall_at_20:.4f}, precision@20 = {precision_at_20:.4f}")
table

Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.1429, recall@20 = 0.0000, precision@20 = 0.0000


,score,is_fraud
person_id,,
16052012,-0.056293,False
6598101,-0.055645,False
11721056,-0.043871,False
12508879,-0.042075,False
11684714,-0.042069,False
11730290,-0.040440,False
11750504,-0.036019,False
11663876,-0.033753,False
11129711,-0.032118,False


### 2. використовую усі фічі, але додаю activity_state=2 (клієнт замовляв 3+ рази), а також дні відвідувань 3+
Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.2500, Recall = 0.0000

In [36]:
df, client_data = load_data(activity_state = 2)
client_data = client_data[client_data['days_visits'] > 2]
X_std = scale_features(data=client_data, scaler_type="standard")

features = [
    'num_of_trn',
    'days_visits',
    'gross_amount_mean',
    'gross_amount_sum',
    'bonuses_accum_sum',
    'bonuses_used_sum',
    'num_of_waiters',
    'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'share_top_waiter',
    'share_bonus_trn',
    'share_bonus_after_first',
    'num_of_places',
    'share_top_places',
    'share_bonuses_used_top_waiter'
]
skewed = [
    'num_of_trn',
    'days_visits',
    'gross_amount_mean',
    'gross_amount_sum',
    'bonuses_accum_sum',
    'bonuses_used_sum',
    'num_of_waiters',
    'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'num_of_places'
]

X_std2 =  scale_features(
        data = client_data,
        scaler_type = "standard",
        features = features,
        skewed = skewed,
        show_charts = False,
        fit_data = None,
)
iso = IsolationForest(
    n_estimators=100,
    contamination=0.001,
    random_state=42
)

train_data = X_std2[~X_std2.index.isin(FRAUD_IDS)]
iso.fit(train_data)

client_data['anomaly_score_std'] = iso.decision_function(X_std2)
client_data['anomaly_label_std'] = iso.predict(X_std2)
hit_rate = client_data[client_data['is_fraud']]["anomaly_label_std"].value_counts(normalize=True).get(-1, 0)

top_20 = client_data['anomaly_score_std'].sort_values().head(20)
fraud_in_top_20 = top_20.index.isin(FRAUD_IDS)
n_fraud = int(client_data['is_fraud'].sum())
# recall@20 = (frauds in top 20) / total_frauds (same definition as models.py)
recall_at_20 = fraud_in_top_20.sum() / n_fraud if n_fraud else np.nan
# precision@20 = (frauds in top 20) / 20 (what the notebook previously called "Recall")
precision_at_20 = fraud_in_top_20.mean()
table = pd.DataFrame({
    'score': top_20.values,
    'is_fraud': fraud_in_top_20
}, index=top_20.index)
print(f"Isolation Forest with contamination=0.001: Anomaly Hit Rate = {hit_rate:.4f}, recall@20 = {recall_at_20:.4f}, precision@20 = {precision_at_20:.4f}")
table

Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.2500, recall@20 = 0.0000, precision@20 = 0.0000


,score,is_fraud
person_id,,
6598101,-0.064189,False
16052012,-0.050535,False
13995241,-0.046093,False
12508879,-0.044359,False
13074462,-0.042389,False
12715006,-0.042183,False
12270045,-0.040474,False
11790326,-0.039402,False
12547600,-0.039371,False


In [13]:
fraud_ids = client_data[client_data.index.isin(top_20.index)].index

# Sort fraud_ids by anomaly score (ascending: most anomalous first)
fraud_ids_sorted = client_data.loc[fraud_ids, 'anomaly_score_std'].sort_values().index

# Collect percentiles for each fraud person_id (for selected features)
train = X_std2[~X_std2.index.isin(FRAUD_IDS)]

features_for_table = ['anomaly_score_std'] + features
rows = []

for person_id in fraud_ids_sorted:
    row = {}
    for feat in features_for_table:
        if feat == 'anomaly_score_std':
            row[feat] = round(client_data.loc[person_id, 'anomaly_score_std'], 4)
        else:
            val = X_std2.loc[person_id, feat]
            # Calculate percentile vs train
            percentile = (train[feat] < val).mean() * 100
            row[feat] = round(percentile, 2)
    rows.append(row)

result_df = pd.DataFrame(rows, index=fraud_ids_sorted)
result_df.index.name = "person_id"
result_df.to_csv('res_df.csv')
result_df

,anomaly_score_std,num_of_trn,days_visits,gross_amount_mean,gross_amount_sum,bonuses_accum_sum,bonuses_used_sum,num_of_waiters,gross_amount_max,first_last_trn_diff,first_second_trn_diff,first_third_trn_diff,time_between_trn_median,trn_per_day,share_top_waiter,share_bonus_trn,share_bonus_after_first,num_of_places,share_top_places,share_bonuses_used_top_waiter
person_id,,,,,,,,,,,,,,,,,,,,
6598101,-0.0642,99.97,98.98,53.93,99.99,100.00,100.00,97.19,99.85,18.86,54.34,33.65,0.23,99.97,30.93,99.82,97.42,0.00,85.39,61.08
16052012,-0.0505,99.90,93.45,54.30,99.97,99.97,99.96,0.00,97.95,13.37,13.39,5.98,2.81,100.00,99.26,50.01,43.27,0.00,85.39,91.86
13995241,-0.0461,10.00,0.00,1.13,0.65,0.16,21.38,0.00,1.17,1.18,29.29,20.23,24.64,47.68,99.26,40.51,48.51,0.00,85.39,91.86
12508879,-0.0444,100.00,99.95,5.82,99.99,99.99,99.99,92.98,36.45,66.92,11.37,17.91,4.10,99.63,20.32,99.82,97.44,14.61,85.34,58.82
13074462,-0.0424,99.56,99.80,0.11,70.57,0.43,15.73,0.74,0.04,50.82,47.00,29.18,29.57,0.00,99.25,14.87,14.96,0.00,85.39,91.86
12715006,-0.0422,10.00,0.00,0.07,0.04,0.13,0.00,0.00,0.02,1.73,11.25,18.60,28.40,47.68,99.26,0.00,0.00,0.00,85.39,41.94
12270045,-0.0405,21.16,0.00,0.26,0.37,1.23,16.55,0.00,0.18,12.77,11.73,4.37,58.44,70.82,99.26,91.00,93.93,0.00,85.39,91.86
11790326,-0.0394,21.16,25.68,0.02,0.03,0.57,16.96,0.00,0.19,3.86,41.17,26.23,28.46,40.04,99.26,31.57,35.61,0.00,85.39,91.86
12547600,-0.0394,96.16,97.70,0.05,21.02,0.16,15.46,0.00,0.15,49.27,38.94,28.86,40.22,26.53,99.26,14.93,15.03,0.00,85.39,91.86


з таблиці зверху видно, що в основному картки отримують нижчий скор через аномально малі/низькі суми чеків, накобичених бофонів і тд. незважаючи на заскейлені дані, ці фічі беруться пріоритетніше

### 3. відкидаю певні фічі, а саме більшість абсолютних метрик  
- num_of_trn
- days_visits
- gross_amount_sum
- bonuses_accum_sum
- bonuses_used_sum
- num_of_waiters
- gross_amount_max
- num_of_places  

за рахунок цих метрик багато карток афектяться як аномалії, хоча насправді це можуть бути старі клієнти з великою базою транзакцій. для них фічі по типу великих сум чеків, к-ті транзакцій і тд будуть пріоритетнішими за рейтові метрики, хоча в даному контексті вони цінності не несуть.  

Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.2143, recall@20 = 0.0357, precision@20 = 0.0500

In [37]:
df, client_data = load_data(activity_state = 2)
client_data = client_data[client_data['days_visits'] > 2]
X_std = scale_features(data=client_data, scaler_type="standard")

features = [
    # 'num_of_trn',
    # 'days_visits',
    'gross_amount_mean',
    # 'gross_amount_sum',
    # 'bonuses_accum_sum',
    # 'bonuses_used_sum',
    # 'num_of_waiters',
    # 'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'share_top_waiter',
    'share_bonus_trn',
    'share_bonus_after_first',
    # 'num_of_places',
    'share_top_places',
    'share_bonuses_used_top_waiter'
]
skewed = [
    # 'num_of_trn',
    # 'days_visits',
    'gross_amount_mean',
    # 'gross_amount_sum',
    # 'bonuses_accum_sum',
    # 'bonuses_used_sum',
    # 'num_of_waiters',
    # 'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    # 'num_of_places'
]

X_std2 =  scale_features(
        data = client_data,
        scaler_type = "standard",
        features = features,
        skewed = skewed,
        show_charts = False,
        fit_data = None,
)
iso = IsolationForest(
    n_estimators=100,
    contamination=0.001,
    random_state=42
)

train_data = X_std2[~X_std2.index.isin(FRAUD_IDS)]
iso.fit(train_data)

client_data['anomaly_score_std'] = iso.decision_function(X_std2)
client_data['anomaly_label_std'] = iso.predict(X_std2)
hit_rate = client_data[client_data['is_fraud']]["anomaly_label_std"].value_counts(normalize=True).get(-1, 0)

top_20 = client_data['anomaly_score_std'].sort_values().head(20)
fraud_in_top_20 = top_20.index.isin(FRAUD_IDS)
n_fraud = int(client_data['is_fraud'].sum())
# recall@20 = (frauds in top 20) / total_frauds (same definition as models.py)
recall_at_20 = fraud_in_top_20.sum() / n_fraud if n_fraud else np.nan
# precision@20 = (frauds in top 20) / 20 (what the notebook previously called "Recall")
precision_at_20 = fraud_in_top_20.mean()
table = pd.DataFrame({
    'score': top_20.values,
    'is_fraud': fraud_in_top_20
}, index=top_20.index)
print(f"Isolation Forest with contamination=0.001: Anomaly Hit Rate = {hit_rate:.4f}, recall@20 = {recall_at_20:.4f}, precision@20 = {precision_at_20:.4f}")
table

Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.2143, recall@20 = 0.0357, precision@20 = 0.0500


,score,is_fraud
person_id,,
12231243,-0.051868,False
11667048,-0.050155,False
16234805,-0.047609,False
15595950,-0.046416,False
12380638,-0.046385,False
11876561,-0.042187,False
11962560,-0.041730,False
13099115,-0.041467,False
16846666,-0.040801,True


In [15]:
fraud_ids = client_data[client_data.index.isin(top_20.index)].index

# Sort fraud_ids by anomaly score (ascending: most anomalous first)
fraud_ids_sorted = client_data.loc[fraud_ids, 'anomaly_score_std'].sort_values().index

# Collect percentiles for each fraud person_id (for selected features)
train = X_std2[~X_std2.index.isin(FRAUD_IDS)]

features_for_table = ['anomaly_score_std'] + features
rows = []

for person_id in fraud_ids_sorted:
    row = {}
    for feat in features_for_table:
        if feat == 'anomaly_score_std':
            row[feat] = round(client_data.loc[person_id, 'anomaly_score_std'], 4)
        else:
            val = X_std2.loc[person_id, feat]
            # Calculate percentile vs train
            percentile = (train[feat] < val).mean() * 100
            row[feat] = round(percentile, 2)
    rows.append(row)

result_df = pd.DataFrame(rows, index=fraud_ids_sorted)
result_df.index.name = "person_id"
result_df.to_csv('res_df.csv')
result_df

,anomaly_score_std,gross_amount_mean,first_last_trn_diff,first_second_trn_diff,first_third_trn_diff,time_between_trn_median,trn_per_day,share_top_waiter,share_bonus_trn,share_bonus_after_first,share_top_places,share_bonuses_used_top_waiter
person_id,,,,,,,,,,,,
12231243,-0.0519,4.45,2.47,5.51,21.43,13.90,90.37,99.26,97.01,96.64,85.39,91.86
11667048,-0.0502,10.16,5.62,13.85,2.83,1.01,100.00,99.12,15.38,15.43,85.39,91.57
16234805,-0.0476,2.20,5.23,13.08,2.42,0.92,90.37,98.96,97.01,96.64,85.39,91.57
15595950,-0.0464,23.22,3.66,9.93,15.29,15.64,92.87,99.26,98.85,97.29,85.39,91.86
12380638,-0.0464,47.17,0.93,35.94,23.53,26.53,0.00,99.26,99.83,97.44,85.39,91.86
11876561,-0.0422,5.23,0.98,37.19,23.59,26.63,0.00,99.26,94.00,97.44,85.39,91.86
11962560,-0.0417,1.20,1.55,41.84,24.50,28.11,0.00,99.26,94.00,97.44,85.39,91.86
13099115,-0.0415,4.87,26.08,68.98,47.89,4.26,82.21,99.26,99.37,97.44,85.39,91.86
16846666,-0.0408,19.60,6.38,28.74,9.59,7.18,99.96,99.26,17.34,17.06,85.39,91.86


In [16]:
fraud_ids = client_data[client_data['is_fraud'] == 1].index

# Sort fraud_ids by anomaly score (ascending: most anomalous first)
fraud_ids_sorted = client_data.loc[fraud_ids, 'anomaly_score_std'].sort_values().index

# Collect percentiles for each fraud person_id (for selected features)
train = X_std2[~X_std2.index.isin(FRAUD_IDS)]

features_for_table = ['anomaly_score_std'] + features
rows = []

for person_id in fraud_ids_sorted:
    row = {}
    for feat in features_for_table:
        if feat == 'anomaly_score_std':
            row[feat] = round(client_data.loc[person_id, 'anomaly_score_std'], 4)
        else:
            val = X_std2.loc[person_id, feat]
            # Calculate percentile vs train
            percentile = (train[feat] < val).mean() * 100
            row[feat] = round(percentile, 2)
    rows.append(row)

result_df = pd.DataFrame(rows, index=fraud_ids_sorted)
result_df.index.name = "person_id"
result_df.to_csv('res_df.csv')
result_df

,anomaly_score_std,gross_amount_mean,first_last_trn_diff,first_second_trn_diff,first_third_trn_diff,time_between_trn_median,trn_per_day,share_top_waiter,share_bonus_trn,share_bonus_after_first,share_top_places,share_bonuses_used_top_waiter
person_id,,,,,,,,,,,,
16846666,-0.0408,19.60,6.38,28.74,9.59,7.18,99.96,99.26,17.34,17.06,85.39,91.86
16376555,-0.0233,38.66,9.86,20.24,5.98,4.85,99.94,99.26,31.24,28.31,85.39,91.86
16794470,-0.0149,43.45,6.51,27.82,10.54,9.49,99.91,99.26,54.63,48.00,85.39,91.86
16921722,-0.0134,50.79,5.73,19.78,26.43,4.88,99.88,99.26,48.15,35.51,85.39,91.86
11968000,-0.0015,92.63,18.12,2.43,0.32,11.00,99.07,98.95,19.16,18.56,81.07,0.00
11970409,-0.0012,95.11,7.19,9.70,1.16,12.34,98.34,98.81,26.19,26.72,79.71,0.00
12396334,0.0135,90.51,7.18,19.17,9.99,19.72,81.85,99.15,15.32,15.43,84.70,0.00
11766135,0.0138,65.54,92.88,70.16,74.29,9.93,96.13,99.18,25.85,23.82,85.39,91.72
13969910,0.0287,37.47,36.26,1.51,0.62,6.38,99.38,94.83,25.83,23.78,85.32,88.52


### 4. відкидаю фічу share_bonus_trn
порівнюючи дві таблиці зверху бачимо, що аномаліями детектяться переважно картки, де висока частка share_bonus_trn. в той час як на реальних фрод транзакціях це не так однозначно. половина карток не так сильно відхиляється саме по share_bonus_trn

top20 recall 5% -> 20% 

One Class SVM: Anomaly Hit Rate = 0.2143, recall@20 = 0.1429, precision@20 = 0.2000

In [82]:
df, client_data = load_data(activity_state = 2)
client_data = client_data[client_data['days_visits'] > 2]

features = [
    'bonus_trn_count',
    'share_top_waiter',
    'share_bonuses_used_top_waiter',
    'share_top_places',
    'num_of_trn_prcnt',
    'days_visits_prcnt',
    'gross_amount_mean_prcnt',
    'gross_amount_sum_prcnt',
    'bonuses_accum_sum_prcnt',
    'bonuses_used_sum_prcnt',
    'num_of_waiters_prcnt',
    'gross_amount_max_prcnt',
    'first_last_trn_diff_prcnt',
    'first_second_trn_diff_prcnt',
    'first_third_trn_diff_prcnt',
    'time_between_trn_median_prcnt',
    'trn_per_day_prcnt',
    'num_of_places_prcnt'
]
skewed = [
    'bonus_trn_count'
]
table, df_iso, top_20, X_std = test_model(client_data, features, skewed, model='iso')
table

Starting test_model
One Class SVM: Anomaly Hit Rate = 0.3571, recall@20 = 0.1429, precision@20 = 0.2000


,score,is_fraud
person_id,,
16052012,-0.063823,False
16118378,-0.055221,False
12296587,-0.050657,False
11112614,-0.045785,False
11974519,-0.043992,False
16794470,-0.040434,True
11766135,-0.037322,True
14025190,-0.035908,False
15136877,-0.035769,False


In [83]:
check_top_anomaly(df_iso, X_std, top_20, features)

,anomaly_score_std,bonus_trn_count,share_top_waiter,share_bonuses_used_top_waiter,share_top_places,num_of_trn_prcnt,days_visits_prcnt,gross_amount_mean_prcnt,gross_amount_sum_prcnt,bonuses_accum_sum_prcnt,bonuses_used_sum_prcnt,num_of_waiters_prcnt,gross_amount_max_prcnt,first_last_trn_diff_prcnt,first_second_trn_diff_prcnt,first_third_trn_diff_prcnt,time_between_trn_median_prcnt,trn_per_day_prcnt,num_of_places_prcnt
person_id,,,,,,,,,,,,,,,,,,,
16052012,-0.0638,99.84,99.26,91.86,85.39,99.90,93.45,54.30,99.97,99.97,99.96,0.00,97.95,13.37,13.39,5.98,2.81,100.00,0.00
16118378,-0.0552,98.69,99.26,91.86,85.39,98.71,92.87,10.96,92.02,93.67,94.65,0.00,25.01,12.37,20.83,6.29,4.21,98.52,0.00
12296587,-0.0507,97.07,99.25,91.86,85.27,98.06,84.26,25.15,94.30,97.23,97.87,0.74,51.92,10.26,17.23,4.35,4.21,99.63,14.61
11112614,-0.0458,99.39,99.26,91.86,85.39,99.37,98.62,7.56,94.96,90.95,93.16,0.00,74.71,24.83,22.32,7.03,17.35,87.39,0.00
11974519,-0.0440,99.83,99.12,91.86,85.39,99.86,99.75,9.76,99.35,99.41,99.49,0.74,30.33,77.96,14.79,4.24,26.64,80.82,0.00
16794470,-0.0404,97.07,99.26,91.86,85.39,97.44,72.87,43.45,96.46,98.42,97.76,0.00,91.78,6.51,27.82,10.54,9.49,99.91,0.00
11766135,-0.0373,94.93,99.18,91.72,85.39,98.81,95.45,65.54,99.59,98.90,98.73,0.74,92.28,92.88,70.16,74.29,9.93,96.13,0.00
14025190,-0.0359,97.07,99.20,78.55,85.39,99.67,72.87,11.14,98.30,99.70,99.34,0.74,45.37,6.36,20.24,4.75,1.69,100.00,0.00
15136877,-0.0358,99.74,98.45,91.04,85.39,99.89,97.18,43.14,99.94,99.89,99.79,0.74,90.25,22.23,48.18,34.32,2.81,99.96,0.00


In [55]:
check_real_frauds(df_iso, X_std, top_20, features)

,anomaly_score_std,gross_amount_mean,first_last_trn_diff,first_second_trn_diff,first_third_trn_diff,time_between_trn_median,trn_per_day,share_top_waiter,share_top_places,share_bonuses_used_top_waiter
person_id,,,,,,,,,,
16846666,-0.0492,19.60,6.38,28.74,9.59,7.18,99.96,99.26,85.39,91.86
16376555,-0.0491,38.66,9.86,20.24,5.98,4.85,99.94,99.26,85.39,91.86
16794470,-0.0464,43.45,6.51,27.82,10.54,9.49,99.91,99.26,85.39,91.86
16921722,-0.0447,50.79,5.73,19.78,26.43,4.88,99.88,99.26,85.39,91.86
11970409,-0.0066,95.11,7.19,9.70,1.16,12.34,98.34,98.81,79.71,0.00
11968000,-0.0000,92.63,18.12,2.43,0.32,11.00,99.07,98.95,81.07,0.00
11766135,0.0027,65.54,92.88,70.16,74.29,9.93,96.13,99.18,85.39,91.72
12396334,0.0250,90.51,7.18,19.17,9.99,19.72,81.85,99.15,84.70,0.00
13969910,0.0397,37.47,36.26,1.51,0.62,6.38,99.38,94.83,85.32,88.52


### OCSVM

#### 1st try

In [86]:
df, client_data = load_data(activity_state = 2)
client_data = client_data[client_data['days_visits'] > 2]

features = [
    'bonus_trn_count',
    'share_top_waiter',
    'share_bonuses_used_top_waiter',
    'share_top_places',
    'num_of_trn_prcnt',
    'days_visits_prcnt',
    'gross_amount_mean_prcnt',
    'gross_amount_sum_prcnt',
    'bonuses_accum_sum_prcnt',
    'bonuses_used_sum_prcnt',
    'num_of_waiters_prcnt',
    'gross_amount_max_prcnt',
    'first_last_trn_diff_prcnt',
    'first_second_trn_diff_prcnt',
    'first_third_trn_diff_prcnt',
    'time_between_trn_median_prcnt',
    'trn_per_day_prcnt',
    'num_of_places_prcnt'
]
skewed = [
    'bonus_trn_count'
]
table, df_ocsvm, top_20, X_std = test_model(client_data, features, skewed, model='ocsvm')
table

Starting test_model
One Class SVM: Anomaly Hit Rate = 0.5714, recall@20 = 0.0714, precision@20 = 0.1000


,score,is_fraud
person_id,,
11808086,-86.816420,False
6598101,-76.760303,False
12508879,-74.218252,False
16052012,-71.548590,False
12397778,-64.760196,False
11833164,-62.345809,False
11858681,-58.332740,False
11968000,-57.081992,True
11763106,-51.453200,False


In [67]:
check_top_anomaly(df_ocsvm, X_std, top_20, features)

,anomaly_score_std,bonus_trn_count,share_bonus_after_first,share_bonus_trn,share_top_waiter,share_bonuses_used_top_waiter,share_top_places,num_of_trn_prcnt,days_visits_prcnt,gross_amount_mean_prcnt,...,bonuses_accum_sum_prcnt,bonuses_used_sum_prcnt,num_of_waiters_prcnt,gross_amount_max_prcnt,first_last_trn_diff_prcnt,first_second_trn_diff_prcnt,first_third_trn_diff_prcnt,time_between_trn_median_prcnt,trn_per_day_prcnt,num_of_places_prcnt
person_id,,,,,,,,,,,,,,,,,,,,,
12508879,-76.3948,100.00,97.44,99.82,20.32,58.82,85.34,100.00,99.95,5.82,...,99.99,99.99,92.98,36.45,66.92,11.37,17.91,4.10,99.63,14.61
6598101,-76.1535,100.00,97.42,99.82,30.93,61.08,85.39,99.97,98.98,53.93,...,100.00,100.00,97.19,99.85,18.86,54.34,33.65,0.23,99.97,0.00
11808086,-73.9534,100.00,85.09,89.54,98.77,91.70,83.37,100.00,99.99,2.15,...,99.98,99.99,89.48,54.43,90.66,44.08,28.41,12.44,97.80,74.79
11690616,-63.2456,99.75,97.44,99.83,9.84,41.83,45.54,97.52,96.85,80.13,...,6.81,100.00,98.61,76.56,17.68,14.16,16.57,25.37,69.90,97.68
16052012,-61.2177,99.84,43.27,50.01,99.26,91.86,85.39,99.90,93.45,54.30,...,99.97,99.96,0.00,97.95,13.37,13.39,5.98,2.81,100.00,0.00
11750504,-56.6046,99.99,97.44,99.83,97.36,91.04,85.39,99.90,99.96,39.63,...,99.94,99.95,73.24,50.04,94.29,57.47,50.36,30.26,0.00,0.00
12397778,-54.9292,0.00,0.00,0.00,99.17,41.94,83.63,68.35,25.68,94.18,...,95.27,0.00,0.74,99.48,51.97,97.66,93.52,0.10,96.27,14.61
11833164,-53.2442,0.00,0.00,0.00,7.69,41.94,85.39,91.88,53.53,99.99,...,100.00,0.00,89.48,99.96,86.68,4.07,85.30,0.07,99.63,0.00
11858681,-51.3733,0.00,0.00,0.00,99.26,41.94,85.39,21.16,0.00,96.96,...,82.75,0.00,0.00,94.42,76.78,4.36,0.43,64.96,70.82,0.00


In [62]:
check_real_frauds(df_ocsvm, X_std, top_20, features)

,anomaly_score_std,bonus_trn_count,share_bonus_after_first,share_bonus_trn,share_top_waiter,share_bonuses_used_top_waiter,share_top_places,num_of_trn_prcnt,days_visits_prcnt,gross_amount_mean_prcnt,...,bonuses_accum_sum_prcnt,bonuses_used_sum_prcnt,num_of_waiters_prcnt,gross_amount_max_prcnt,first_last_trn_diff_prcnt,first_second_trn_diff_prcnt,first_third_trn_diff_prcnt,time_between_trn_median_prcnt,trn_per_day_prcnt,num_of_places_prcnt
person_id,,,,,,,,,,,,,,,,,,,,,
11968000,-42.2721,89.14,18.56,19.16,98.95,0.00,81.07,98.71,91.53,92.63,...,99.98,99.83,73.24,93.42,18.12,2.43,0.32,11.00,99.07,74.79
11766135,-31.3580,94.93,23.82,25.85,99.18,91.72,85.39,98.81,95.45,65.54,...,98.90,98.73,0.74,92.28,92.88,70.16,74.29,9.93,96.13,0.00
12396334,-29.0435,14.86,15.43,15.32,99.15,0.00,84.70,86.58,79.71,90.51,...,99.37,96.22,4.51,91.20,7.18,19.17,9.99,19.72,81.85,14.61
11970409,-27.5856,67.75,26.72,26.19,98.81,0.00,79.71,89.30,61.84,95.11,...,99.58,99.68,32.61,95.88,7.19,9.70,1.16,12.34,98.34,49.45
16846666,-24.7486,67.75,17.06,17.34,99.26,91.86,85.39,96.90,61.84,19.60,...,95.98,91.41,0.00,31.40,6.38,28.74,9.59,7.18,99.96,0.00
16794470,-23.3985,97.07,48.00,54.63,99.26,91.86,85.39,97.44,72.87,43.45,...,98.42,97.76,0.00,91.78,6.51,27.82,10.54,9.49,99.91,0.00
16376555,-19.9329,86.27,28.31,31.24,99.26,91.86,85.39,95.15,53.53,38.66,...,91.04,95.93,0.00,77.09,9.86,20.24,5.98,4.85,99.94,0.00
12861171,-14.7904,98.98,68.76,77.66,98.80,91.71,85.39,98.17,99.04,98.98,...,99.98,99.98,4.51,97.74,36.83,44.76,27.03,38.29,25.08,0.00
16921722,-13.7381,91.25,35.51,48.15,99.26,91.86,85.39,94.33,53.53,50.79,...,94.38,96.32,0.00,59.12,5.73,19.78,26.43,4.88,99.88,0.00


#### 2nd try

In [87]:
df, client_data = load_data(activity_state = 2)
client_data = client_data[client_data['days_visits'] > 2]

features = [
    'bonus_trn_count',
    'share_top_waiter',
    'share_bonuses_used_top_waiter',
    'share_top_places',
    'num_of_trn_prcnt',
    'days_visits_prcnt',
    'gross_amount_mean_prcnt',
    'gross_amount_sum_prcnt',
    'bonuses_accum_sum_prcnt',
    'bonuses_used_sum_prcnt',
    'num_of_waiters_prcnt',
    'gross_amount_max_prcnt',
    'first_last_trn_diff_prcnt',
    'first_second_trn_diff_prcnt',
    'first_third_trn_diff_prcnt',
    'time_between_trn_median_prcnt',
    'trn_per_day_prcnt',
    'num_of_places_prcnt'
]
skewed = [
    'bonus_trn_count'
]
table, df_ocsvm, top_20, X_std = test_model(client_data, features, skewed, model='ocsvm')
table

Starting test_model
One Class SVM: Anomaly Hit Rate = 0.5714, recall@20 = 0.0714, precision@20 = 0.1000


,score,is_fraud
person_id,,
11808086,-86.816420,False
6598101,-76.760303,False
12508879,-74.218252,False
16052012,-71.548590,False
12397778,-64.760196,False
11833164,-62.345809,False
11858681,-58.332740,False
11968000,-57.081992,True
11763106,-51.453200,False
